In [1]:
# ===== CELL 0: inputs =====
import os, json, glob
from pathlib import Path

GT_JSON = next(p for p in Path("/kaggle/input").rglob("*.json") if "video_tracking" in p.name)
print("GT json:", GT_JSON.name, f"| {GT_JSON.stat().st_size/1e6:.0f} MB")

videos = sorted(Path("/kaggle/input").rglob("*.mp4"))
print(f"\nvideos attached: {len(videos)}")
for v in videos:
    print("  ", v.name, f"| {v.stat().st_size/1e6:.0f} MB")

# purane crops ka root (agar attached hai)
OLD_ROOT = None
for dirpath, dirnames, _ in os.walk("/kaggle/input"):
    if any(d.lower().startswith("cam_") for d in dirnames):
        OLD_ROOT = dirpath
        break
print("\npurane crops ka root:", OLD_ROOT)

OUT = "/kaggle/working/new_crops"
os.makedirs(OUT, exist_ok=True)
print("output:", OUT)

GT json: tracking_db.video_tracking (2).json | 100 MB

videos attached: 6
   Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_017.mp4 | 416 MB
   Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_018.mp4 | 424 MB
   Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_023.mp4 | 512 MB
   Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_026.mp4 | 356 MB
   Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_031.mp4 | 331 MB
   Cam_1_20260222T000000Z_20260223T000000Z_10fps_chunk_027.mp4 | 478 MB

purane crops ka root: /kaggle/input/datasets/payal1238gupta/cam-12-chunk-0-12
output: /kaggle/working/new_crops


In [2]:
# ===== CELL 1: coverage =====
from collections import Counter

db = json.load(open(GT_JSON))
print("total frame-entries:", len(db))

ann = Counter(e["video"] for e in db)
print("annotated chunks in JSON:", len(ann))

have_crops = set()
if OLD_ROOT:
    have_crops = {d for d in os.listdir(OLD_ROOT) if d.lower().startswith("cam_")}
print("chunks with crops already:", len(have_crops))

have_video = {v.stem: v for v in videos}
print("chunks with video attached:", len(have_video))

TEST_KEY = next((k for k in ann if "chunk_032" in k), None)
print("TEST chunk (skip hoga):", TEST_KEY)

todo = []
print("\n%-12s %8s %8s %6s %6s" % ("chunk", "frames", "tracks", "crops?", "video?"))
for key in sorted(ann):
    short = key.split("chunk_")[-1] if "chunk_" in key else key[-6:]
    tracks = len({t["track_id"] for e in db if e["video"] == key for t in e.get("tracks", [])})
    c = "yes" if key in have_crops else "-"
    v = "yes" if key in have_video else "-"
    print("chunk_%-6s %8d %8d %6s %6s" % (short, ann[key], tracks, c, v))
    if key not in have_crops and key in have_video and key != TEST_KEY:
        todo.append(key)

print("\nEXTRACT karne layak chunks:", len(todo))
for k in todo:
    print("  ", k)
if not todo:
    print("\n>> kuch nahi mila. Matlab jin chunks ke crops nahi hain, unke videos attached nahi hain.")
    print("   Drive se wo videos Kaggle pe upload karo, phir ye notebook dobara chalao.")

total frame-entries: 230649
annotated chunks in JSON: 31
chunks with crops already: 15
chunks with video attached: 6
TEST chunk (skip hoga): Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_032

chunk          frames   tracks crops? video?
chunk_001        7761      134    yes      -
chunk_002        7030      237    yes      -
chunk_003        4399      124    yes      -
chunk_004        1274       21    yes      -
chunk_006         228        3    yes      -
chunk_007         926       20    yes      -
chunk_008         167        9    yes      -
chunk_009         212       11    yes      -
chunk_010         916       16    yes      -
chunk_011        1198       32    yes      -
chunk_012        4635       49    yes      -
chunk_013       19709       42    yes      -
chunk_014        8598       65    yes      -
chunk_015        6384       83    yes      -
chunk_016       10403      105    yes      -
chunk_017        4674      100      -    yes
chunk_018        4504       88      

In [3]:
# ===== CELL 2: extraction (fast, sequential read) =====
import cv2, time

POSTER_BOX = [385, 84, 622, 381]
POSTER_THR = 0.95
MIN_W, MIN_H = 40, 80

def in_poster(x1, y1, x2, y2):
    px1, py1, px2, py2 = POSTER_BOX
    ix1, iy1 = max(x1, px1), max(y1, py1)
    ix2, iy2 = min(x2, px2), min(y2, py2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    return inter / max((x2 - x1) * (y2 - y1), 1) > POSTER_THR

total_saved = 0

for key in todo:
    vid = have_video[key]
    by_frame = {}
    for e in db:
        if e["video"] == key and e.get("tracks"):
            by_frame[e["frame_number"]] = e["tracks"]
    if not by_frame:
        continue
    last = max(by_frame)

    cap = cv2.VideoCapture(str(vid))
    t0, saved, fno = time.time(), 0, 0
    while fno <= last:
        ok = cap.grab()                      # decode nahi, sirf aage badho
        if not ok:
            break
        if fno in by_frame:
            ok, frame = cap.retrieve()       # sirf zaroori frame decode karo
            if ok:
                H, W = frame.shape[:2]
                for t in by_frame[fno]:
                    x1, y1, x2, y2 = [int(round(v)) for v in t["bbox"]]
                    if in_poster(x1, y1, x2, y2):
                        continue
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(W, x2), min(H, y2)
                    if x2 - x1 < MIN_W or y2 - y1 < MIN_H:
                        continue
                    d = os.path.join(OUT, key, str(t["track_id"]))
                    os.makedirs(d, exist_ok=True)
                    cv2.imwrite(os.path.join(d, f"{fno}.jpg"),
                                frame[y1:y2, x1:x2], [cv2.IMWRITE_JPEG_QUALITY, 92])
                    saved += 1
        fno += 1
    cap.release()
    total_saved += saved
    print("%-55s | %5d crops | %.1f min" % (key, saved, (time.time() - t0) / 60))

print("\nkul crops:", total_saved)

Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_017 |  6762 crops | 1.0 min
Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_018 |  5972 crops | 1.0 min
Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_023 |  5457 crops | 1.2 min
Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_026 | 11463 crops | 1.0 min
Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_031 |  5850 crops | 0.9 min

kul crops: 35504


In [4]:
# ===== CELL 3: verify =====
from glob import glob as g

chunks = sorted(d for d in os.listdir(OUT)) if os.path.exists(OUT) else []
print("naye chunks:", len(chunks))

ids, imgs = 0, 0
counts = []
for ch in chunks:
    for tid in os.listdir(os.path.join(OUT, ch)):
        n = len(g(os.path.join(OUT, ch, tid, "*.jpg")))
        if n:
            ids += 1
            imgs += n
            counts.append(n)

print("naye identities:", ids, "| naye crops:", imgs)
if counts:
    counts.sort()
    print("images/id  min:", counts[0], "median:", counts[len(counts)//2], "max:", counts[-1])
    print("MIN_IMG=5 ke baad bachengi:", sum(1 for c in counts if c >= 5))

print("\npurani training: 294 ids (236 train)")
print("naye milake    :", 294 + sum(1 for c in counts if c >= 5), "ids (approx)")

naye chunks: 5
naye identities: 513 | naye crops: 35504
images/id  min: 1 median: 36 max: 2422
MIN_IMG=5 ke baad bachengi: 463

purani training: 294 ids (236 train)
naye milake    : 757 ids (approx)


In [5]:
# ===== CELL 4: zip =====
import shutil

if ids:
    shutil.make_archive("/kaggle/working/new_reid_crops", "zip", OUT)
    z = Path("/kaggle/working/new_reid_crops.zip")
    print("bana:", z, f"| {z.stat().st_size/1e6:.0f} MB")
    print("\nab: Output tab se ye zip download karo -> Kaggle pe New Dataset banao -> upload")
else:
    print("kuch extract nahi hua, zip skip")

bana: /kaggle/working/new_reid_crops.zip | 666 MB

ab: Output tab se ye zip download karo -> Kaggle pe New Dataset banao -> upload
